In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("game.csv")
print(df.shape)
df.head()


(65698, 55)


,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,reb_away,ast_away,stl_away,blk_away,tov_away,pf_away,pts_away,plus_minus_away,video_available_away,season_type
0,21946,1610610035,HUS,Toronto Huskies,24600001,1946-11-01 00:00:00,HUS vs. NYK,L,0,25.0,...,NaN,NaN,NaN,NaN,NaN,NaN,68.0,2,0,Regular Season
1,21946,1610610034,BOM,St. Louis Bombers,24600003,1946-11-02 00:00:00,BOM vs. PIT,W,0,20.0,...,NaN,NaN,NaN,NaN,NaN,25.0,51.0,-5,0,Regular Season
2,21946,1610610032,PRO,Providence Steamrollers,24600002,1946-11-02 00:00:00,PRO vs. BOS,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,53.0,-6,0,Regular Season
3,21946,1610610025,CHS,Chicago Stags,24600004,1946-11-02 00:00:00,CHS vs. NYK,W,0,21.0,...,NaN,NaN,NaN,NaN,NaN,22.0,47.0,-16,0,Regular Season
4,21946,1610610028,DEF,Detroit Falcons,24600005,1946-11-02 00:00:00,DEF vs. WAS,L,0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,50.0,17,0,Regular Season


In [2]:
target = "wl_home"

print("Shape:", df.shape)
print("\nTipos:\n", df.dtypes.value_counts())
print("\nNulos top 25:\n", df.isna().sum().sort_values(ascending=False).head(25))

print("\nBalance target:\n", df[target].value_counts())
print("\nBalance %:\n", df[target].value_counts(normalize=True))

df.describe(include="all").T.head(30)


Shape: (65698, 55)

Tipos:
 float64    36
str        10
int64       9
Name: count, dtype: int64

Nulos top 25:
 fg3_pct_home    19074
dreb_home       18999
dreb_away       18998
fg3_pct_away    18962
oreb_away       18936
oreb_home       18936
stl_home        18849
stl_away        18849
tov_away        18685
tov_home        18684
fg3a_away       18683
fg3a_home       18683
blk_home        18626
blk_away        18625
ast_home        15805
ast_away        15801
reb_home        15729
reb_away        15725
fg_pct_home     15490
fg_pct_away     15489
fga_home        15447
fga_away        15447
fg3m_home       13218
fg3m_away       13218
ft_pct_home      3009
dtype: int64

Balance target:
 wl_home
W    40649
L    25047
Name: count, dtype: int64

Balance %:
 wl_home
W    0.618744
L    0.381256
Name: proportion, dtype: float64


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
season_id,65698.0,NaN,NaN,NaN,22949.338747,5000.3055,12005.0,21981.0,21997.0,22011.0,42022.0
team_id_home,65698.0,NaN,NaN,NaN,1609926285.569104,33243133.512716,45.0,1610612744.0,1610612751.0,1610612758.0,1610616834.0
team_abbreviation_home,65698,97,BOS,3124,NaN,NaN,NaN,NaN,NaN,NaN,NaN
team_name_home,65698,98,Boston Celtics,3124,NaN,NaN,NaN,NaN,NaN,NaN,NaN
game_id,65698.0,NaN,NaN,NaN,25847473.129974,6303760.433714,10500001.0,21300533.25,26300068.5,28800693.75,49800087.0
game_date,65698,12882,2009-01-02 00:00:00,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
matchup_home,65698,2292,BOS vs. NYK,258,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wl_home,65696,2,W,40649,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,65698.0,NaN,NaN,NaN,221.003486,67.903521,0.0,240.0,240.0,240.0,365.0
fgm_home,65685.0,NaN,NaN,NaN,39.672269,6.770802,4.0,35.0,40.0,44.0,84.0


In [3]:
target = "wl_home"

drop_cols = ["wl_away", "game_date", "matchup_home", "matchup_away"]
drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=[target] + drop_cols).copy()
y = df[target].astype(str).copy()

valid = y.isin(["W", "L"])
X = X.loc[valid].copy()
y = y.loc[valid].copy()

num_cols_all = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_all = [c for c in X.columns if c not in num_cols_all]

for c in cat_cols_all:
    X[c] = X[c].astype("object")
    X[c] = X[c].where(X[c].notna(), "UNK")
    X[c] = X[c].astype(str)

print("X:", X.shape, "y:", y.shape)
y.value_counts()


X: (65696, 50) y: (65696,)


wl_home
W    40649
L    25047
Name: count, dtype: int64

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent", missing_values="UNK")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000))
])

clf.fit(X_train, y_train)

pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, pos_label="W")
auc = roc_auc_score((y_test == "W").astype(int), proba)

print("Accuracy:", acc)
print("F1 (W):", f1)
print("ROC AUC:", auc)
print("\nMatriz de confusion (W,L):\n", confusion_matrix(y_test, pred, labels=["W","L"]))
print("\nReporte:\n", classification_report(y_test, pred))


Accuracy: 0.9998477929984779
F1 (W): 0.9998769987699877
ROC AUC: 0.9999440970457609

Matriz de confusion (W,L):
 [[8129    1]
 [   1 5009]]

Reporte:
               precision    recall  f1-score   support

           L       1.00      1.00      1.00      5010
           W       1.00      1.00      1.00      8130

    accuracy                           1.00     13140
   macro avg       1.00      1.00      1.00     13140
weighted avg       1.00      1.00      1.00     13140



In [7]:
import joblib, json

joblib.dump(clf, "game_pipeline.joblib")

schema = {
    "target": "wl_home",
    "drop_cols": drop_cols,
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "feature_cols": list(X.columns)
}
with open("schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print("Exportado: game_pipeline.joblib")
print("Exportado: schema.json")


Exportado: game_pipeline.joblib
Exportado: schema.json


In [8]:
import json
import numpy as np
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

string_inputs = [
    "team_abbreviation_home",
    "team_name_home",
    "team_abbreviation_away",
    "team_name_away",
    "season_type"
]

float_inputs = [
  "season_id","team_id_home","game_id","min","fgm_home","fga_home","fg_pct_home",
  "fg3m_home","fg3a_home","fg3_pct_home","ftm_home","fta_home","ft_pct_home",
  "oreb_home","dreb_home","reb_home","ast_home","stl_home","blk_home","tov_home",
  "pf_home","pts_home","plus_minus_home","video_available_home","team_id_away",
  "fgm_away","fga_away","fg_pct_away","fg3m_away","fg3a_away","fg3_pct_away",
  "ftm_away","fta_away","ft_pct_away","oreb_away","dreb_away","reb_away",
  "ast_away","stl_away","blk_away","tov_away","pf_away","pts_away",
  "plus_minus_away","video_available_away"
]

initial_types = []
for n in string_inputs:
    initial_types.append((n, StringTensorType([None, 1])))
for n in float_inputs:
    initial_types.append((n, FloatTensorType([None, 1])))

# IMPORTANTE: desactivar ZipMap para que output_probability sea tensor
options = {id(clf): {"zipmap": False}}

onnx_model = convert_sklearn(
    clf,
    initial_types=initial_types,
    options=options
)

with open("game_pipeline.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

onnx_schema = {"cat_cols": string_inputs, "num_cols": float_inputs}
with open("onnx_schema.json", "w", encoding="utf-8") as f:
    json.dump(onnx_schema, f, ensure_ascii=False, indent=2)

print("ONNX exportado sin ZipMap (Node compatible).")


ONNX exportado sin ZipMap (Node compatible).


In [9]:
import numpy as np
import pandas as pd
import onnxruntime as ort

sess = ort.InferenceSession("game_pipeline.onnx", providers=["CPUExecutionProvider"])

inputs = sess.get_inputs()
input_names = [i.name for i in inputs]
print("Inputs:", input_names)
print("Outputs:", [o.name for o in sess.get_outputs()])

row = X_test.iloc[0].copy()

feeds = {}

for name in input_names:
    if name in cat_cols:
        v = row[name] if name in row.index else "UNK"
        if pd.isna(v):
            v = "UNK"
        feeds[name] = np.array([[str(v)]], dtype=object)
    elif name in num_cols:
        v = row[name] if name in row.index else 0.0
        if pd.isna(v):
            v = 0.0
        feeds[name] = np.array([[np.float32(v)]], dtype=np.float32)
    else:
        feeds[name] = np.array([["UNK"]], dtype=object)

out = sess.run(None, feeds)

print("Numero de outputs:", len(out))
for i, o in enumerate(out):
    print("Output", i, "tipo:", type(o))
    try:
        arr = np.array(o)
        print("shape:", arr.shape)
        flat = arr.ravel()
        print("primeros valores:", flat[:10])
    except Exception as e:
        print("No convertible a numpy directamente:", e)
        print("valor:", o)


Inputs: ['team_abbreviation_home', 'team_name_home', 'team_abbreviation_away', 'team_name_away', 'season_type', 'season_id', 'team_id_home', 'game_id', 'min', 'fgm_home', 'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home', 'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home', 'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home', 'pts_home', 'plus_minus_home', 'video_available_home', 'team_id_away', 'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away', 'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away', 'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away', 'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away']
Outputs: ['label', 'probabilities']
Numero de outputs: 2
Output 0 tipo: <class 'numpy.ndarray'>
shape: (1,)
primeros valores: ['W']
Output 1 tipo: <class 'numpy.ndarray'>
shape: (1, 2)
primeros valores: [0. 1.]
